In [ ]:
import pandas as pd
import numpy as np

Import i wczytanie danych

---



In [ ]:
df_train = pd.read_csv('train.csv', low_memory=False)
df_store = pd.read_csv('store.csv')

# Łączymy tabele po kolumnie 'Store'
df_master = pd.merge(df_train, df_store, how='left', on='Store')

Czyszczenie danych i inżynieria wskaźników biznesowych (KPIs)

In [ ]:
# Odrzucamy dni, w których sklepy były zamknięte lub nie odnotowano sprzedaży
df_master = df_master[(df_master['Open'] == 1) & (df_master['Sales'] > 0)].copy()

# Obsługa brakujących wartości (NaN) w odległości od konkurencji
# Brak wartości oznacza brak konkurencji w pobliżu - wpisujemy bezpieczny, duży dystans
df_master['CompetitionDistance'] = df_master['CompetitionDistance'].fillna(50000)

# Dodanie średniego koszyka
df_master['Basket_Size'] = df_master['Sales'] / df_master['Customers']

# Rozbicie daty
df_master['Date'] = pd.to_datetime(df_master['Date'])
df_master['Year'] = df_master['Date'].dt.year
df_master['Month'] = df_master['Date'].dt.month
df_master['DayOfWeek'] = df_master['Date'].dt.dayofweek + 1

In [ ]:
# Jeśli konkurencja istnieje (mamy dystans), ale nie znamy daty,
# zakładamy biznesowo, że jest tam "od zawsze" (np. od stycznia 1900 roku).
df_master['CompetitionOpenSinceYear'] = df_master['CompetitionOpenSinceYear'].fillna(1900)
df_master['CompetitionOpenSinceMonth'] = df_master['CompetitionOpenSinceMonth'].fillna(1)

# 2. Tworzenie KPI: Czy konkurencja była aktywna w dniu transakcji? (0 = Nie, 1 = Tak)
df_master['Competition_Active'] = 0

# Ustawiamy flagę na 1, jeśli rok transakcji jest większy niż rok otwarcia konkurencji
# LUB jeśli to ten sam rok, ale miesiąc transakcji jest większy bądź równy miesiącowi otwarcia
df_master.loc[(df_master['Year'] > df_master['CompetitionOpenSinceYear']) |
              ((df_master['Year'] == df_master['CompetitionOpenSinceYear']) &
               (df_master['Month'] >= df_master['CompetitionOpenSinceMonth'])), 'Competition_Active'] = 1

# 3. Zabezpieczenie dla sklepów bez konkurencji
# Jeśli wcześniej wstawiliśmy dystans 50000 (brak konkurencji), to flaga zawsze wynosi 0
df_master.loc[df_master['CompetitionDistance'] == 50000, 'Competition_Active'] = 0

In [ ]:
# Obsługa braków dla sklepów niebiorących udziału w programie Promo2
# Skoro nie uczestniczą, to rok i tydzień startu promocji nie istnieją - wstawiamy 0
df_master['Promo2SinceWeek'] = df_master['Promo2SinceWeek'].fillna(0)
df_master['Promo2SinceYear'] = df_master['Promo2SinceYear'].fillna(0)

# Interwał promocji (np. "Jan,Apr,Jul,Oct") to tekst.
# Jeśli sklepu tam nie ma, zastępujemy pustkę czytelną etykietą "Brak"
df_master['PromoInterval'] = df_master['PromoInterval'].fillna('Brak')

Symulacja szoków popytowych (Rozkład Normalny) – testowanie odporności biznesu


In [ ]:
# Inicjalizacja kolumny flagi scenariusza
df_master['Is_Demand_Shock'] = False

# Ustawienie stałego ziarna losowości dla zapewnienia powtarzalności wyników (Reproducibility)
np.random.seed(42)

# Losowanie indeksów dla 5% wszystkich dni handlowych, które zostaną objęte anomalią popytową
shock_indices = np.random.choice(
    df_master.index,
    size=int(len(df_master) * 0.05),
    replace=False
)

# Oznaczenie wylosowanych rekordów flagą True
df_master.loc[shock_indices, 'Is_Demand_Shock'] = True

#EKSPORT DO NOWEGO PLIKU CSV

In [ ]:
output_filename = 'rossmann_decision_data.csv'
print(f"6. Eksportowanie gotowego zbioru do pliku {output_filename}...")
df_master.to_csv(output_filename, index=False)

print("\n--- PROCES ZAKOŃCZONY SUKCESEM ---")
print(f"Liczba rekordów w gotowym zbiorze: {len(df_master)}")
print("Rozkład dni w podziale na scenariusze rynkowe:")
print(df_master['Is_Demand_Shock'].value_counts(normalize=True) * 100)

6. Eksportowanie gotowego zbioru do pliku rossmann_decision_data.csv...

--- PROCES ZAKOŃCZONY SUKCESEM ---
Liczba rekordów w gotowym zbiorze: 844338
Rozkład dni w podziale na scenariusze rynkowe:
Is_Demand_Shock
False    95.000107
True      4.999893
Name: proportion, dtype: float64
